In [ ]:
from langchain_aws import ChatBedrockConverse
from langchain_ollama import ChatOllama,OllamaEmbeddings
from typing import TypedDict, Literal,List,Dict,Annotated
from langgraph.graph import StateGraph,START,END,MessagesState
from pydantic import BaseModel,Field
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
import sqlite3
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage,BaseMessage,ToolMessage
import operator
from langgraph.prebuilt.tool_node import ToolNode
from langchain_core.tools import tool
import uuid
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command,interrupt




In [2]:
llm = ChatOllama(model='qwen2.5:1.5b',temperature=0.2,max_token=100)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [3]:
store = InMemoryStore(index={'embed':embeddings,'dims':1536})

In [4]:
# Everything under this namespace belongs to this customer
CUSTOMER_ID = "cust_001"
MEMORY_NAMESPACE = (CUSTOMER_ID, "preferences")

In [ ]:
# pre-seeded long term memories
store.put(
    MEMORY_NAMESPACE,
    "1",
    {
        "text": "I prefer casual and comfortable clothing for everyday wear."
    }
)

store.put(
    MEMORY_NAMESPACE,
    "2",
    {
        "text": "My favorite brands are Nike, Gap, and Uniqlo."
    }
)

store.put(
    MEMORY_NAMESPACE,
    "3",
    {
        "text": "My budget is typically between $50-100 per item."
    }
)

store.put(
    MEMORY_NAMESPACE,
    "4",
    {
        "text": "I usually wear size M tops and 32 inch jeans."
    }
)

In [6]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]
    memories: list[str]

In [15]:
def retrieve_memories(state:State):
    user_query = state['messages'][-1].content
    res = store.search(
        MEMORY_NAMESPACE,
        query=user_query,
        limit=5
    )
    memories = []
    for result in res:
        memories.append(result.value['text'])
    return {'memories':memories}

def generate_response(state:State):
    user_query = state['messages'][-1].content
    memories = state.get('memories',[])
    memory_text = '\n'.join(f' - {memory}' for memory in memories)
    system_prompt = f"""
        You are an e-commerce shopping assistant.
    
        The following are long-term memories about the customer:
    
        {memory_text}
    
        Use these memories together with the current conversation
        to provide personalized fashion recommendations.
    
        Do not claim to remember conversation messages.
        Only use the long-term customer preferences provided above.
    
        Current customer query:
        {user_query}
        """
    res = llm.invoke([
        SystemMessage(content=system_prompt),
        *state['messages']
    ])
    return {'messages':[res]}

def save_memory(state:State):
    user_message = None
    for message in reversed(state['messages']):
        if isinstance(message,HumanMessage):
            user_message = message
            break

    if user_message:
        memory_id = str(uuid.uuid4())
        store.put(
            MEMORY_NAMESPACE,
            memory_id,
            {
                "text":f'Customer inquiry : {user_message}'  
            } 
        )
    return {}




In [16]:
graph = StateGraph(State)
graph.add_node("retrieve_memories",retrieve_memories)
graph.add_node("generate_response",generate_response)
graph.add_node("save_memory",save_memory)

graph.add_edge(START,"retrieve_memories")
graph.add_edge("retrieve_memories","generate_response")
graph.add_edge("generate_response","save_memory")
graph.add_edge("save_memory",END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer,store=store)

In [17]:
print(workflow.get_graph().draw_ascii())

    +-----------+      
    | __start__ |      
    +-----------+      
          *            
          *            
          *            
+-------------------+  
| retrieve_memories |  
+-------------------+  
          *            
          *            
          *            
+-------------------+  
| generate_response |  
+-------------------+  
          *            
          *            
          *            
   +-------------+     
   | save_memory |     
   +-------------+     
          *            
          *            
          *            
     +---------+       
     | __end__ |       
     +---------+       


In [18]:
config_thread_1 = {"configurable": {"thread_id": "thread_1"}}
result_1 = workflow.invoke(
    {
        "messages": [HumanMessage(content="I am looking for running shoes.")]
    },
    config=config_thread_1
)
print("THREAD 1:",result_1["messages"][-1].content)

THREAD 1: Based on your preference for casual and comfortable clothing, you might enjoy some of these running shoe brands: Asics, Brooks, and Saucony. These brands are known for their comfort and support, which should fit well with your style. Additionally, consider the following options within your budget range:

1. **Asics Gel-Kayano 25** - Known for its cushioned midsole and supportive design.
2. **Brooks Ghost 30** - Offers a lightweight and breathable shoe that's perfect for casual runs.
3. **Saucony Ride 4** - Provides good support with a comfortable fit, suitable for various terrains.

These shoes should meet your comfort and style preferences while staying within your budget of $50-100 per item.


In [19]:
config_thread_1 = {"configurable": {"thread_id": "thread_1"}}
result_2 = workflow.invoke(
    {
        "messages": [HumanMessage(content="I prefer Nike running shoes.")]
    },
    config=config_thread_1
)
print("THREAD 1 result 2:",result_2["messages"][-1].content)

THREAD 1 result 2: Absolutely! Here are some specific Nike running shoes that you might enjoy:

1. **Nike Air Max 97** - Known for its iconic design and excellent cushioning, making it perfect for both casual and training runs.
2. **Nike Free Run 5.0** - Offers a smooth ride with minimal impact on your joints, ideal for those who prefer less pressure on their feet.
3. **Nike Air Zoom Pegasus 38** - Provides great support while offering a comfortable fit that's perfect for everyday use.

These shoes are designed to be both stylish and supportive, ensuring you can enjoy running without any discomfort.


In [21]:
config_thread_2 = {"configurable": {"thread_id": "thread_2"}}
result_2 = workflow.invoke(
    {
        "messages": [HumanMessage(content="Suggest some running shoes for me.")]
    },
    config=config_thread_2
)
print("THREAD 2:",result_2["messages"][-1].content)

THREAD 2: Based on your preference for Nike, Gap, and Uniqlo, I recommend checking out these pairs of running shoes:

1. **Nike Air Max 270 Running Shoes** - These are lightweight and provide excellent cushioning, making them perfect for everyday use.
2. **Gap Men's Running Shoes** - The classic style with a modern twist, offering both comfort and durability.
3. **Uniqlo's Run Series Shoes** - Known for their breathable materials and comfortable fit, these shoes are great for casual runs.

These brands should provide you with a good selection that fits your preferences and budget.
